# P01 · Consolidar cinco Excel públicos que describen lo mismo y no se parecen

> **¿Cuántos lugares de carga de la tarjeta bip! hay, dónde están, y qué identifica a cada uno?**

Cinco `.xlsx` del portal de datos abiertos del MTT, licencia CC BY-NC 2.0 verificada. Seis hojas, 1 928 filas, una en la hoja de cerrados.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/bip_lugares_carga/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py bip-metro bip-puntos bip-centros-alto-estandar bip-centros-estandar bip-retail


## 1 · El problema

Para pagar un viaje en el transporte público de Santiago hay que llevar saldo en la
tarjeta bip!, y el saldo se carga en locales físicos. El organismo publica dónde están
esos locales, y lo publica como lo publica un organismo: **cinco archivos de Excel
distintos**, uno por tipo de local, hechos en momentos distintos por personas distintas.

La pregunta que llega es simple y legítima: *¿cuántos lugares de carga hay y dónde
están?* Alguien tiene que contestarla para decidir dónde instalar el próximo, o para
responder una consulta ciudadana, o para cruzarla con la red de transporte.

**Lo que la vuelve difícil no es el volumen.** Son 1 928 filas: cabrían en una pantalla
y media. Lo difícil es que los cinco archivos no se parecen entre sí:

- el **encabezado no está en la primera fila** de ninguno, y está en una distinta en
  cada uno, porque arriba hay un título y filas en blanco;
- los **esquemas no coinciden**: unos traen nueve columnas y otros diez, y el mismo
  campo se llama de dos formas;
- el **tipo de local no está en los datos**: está en el nombre del archivo;
- una hoja describe locales **fuera de servicio**, y si se lee todo junto quedan
  contados como abiertos;
- la columna que parece la clave **no identifica una fila**;
- y la misma ubicación viene **dos veces**, en dos sistemas de coordenadas que no
  coinciden.

Ninguno de esos seis problemas produce un error. Cada uno produce una tabla que se puede
sumar, graficar y presentar, con un número equivocado.

Ese es el trabajo real de quien gestiona datos que no generó: no calcular, sino
**descubrir en qué no se puede confiar antes de calcular**. Y es exactamente lo que se
pide en una evaluación técnica de cuatro horas con Excel y Pandas.


## 2 · La solución que se va a aplicar

El recorrido va de lo que no se puede suponer a lo que sí se puede afirmar, en cinco
bloques.

**Leer lo que no está ordenado** (pasos 1 a 3). Antes de leer, mirar: cuántas hojas hay
y en qué fila empieza cada tabla. La fila del encabezado se **detecta** en lugar de
fijarse, y de cada lectura queda un informe de qué hubo que hacer. Ese informe es lo que
después permite defender el recuento.

**Armonizar lo que no coincide** (pasos 4 a 6). La correspondencia entre el nombre de
cada archivo y el campo del esquema común se **declara** en una tabla, no se adivina por
parecido: un acierto por casualidad hoy es un error mañana. Y al apilar se agrega el tipo
de local, que vive en el nombre del archivo y se perdería para siempre.

**Comprobar lo que se supone** (pasos 7 y 8). Dos suposiciones que todo el mundo hace y
nadie mide: que la columna llamada «código» es una clave, y que los textos que nombran lo
mismo están escritos igual. Las dos son falsas en este archivo, y medirlo cuesta una
línea cada una.

**Responder, y declarar el límite** (pasos 9 y 10). Recién aquí se cuenta por comuna. Y
se mide cuánto del campo de horario existe de verdad, porque de eso depende qué se puede
concluir después.

**Validar sin fuente externa** (pasos 11 y 12). El archivo trae la ubicación dos veces.
Convertir una y compararla con la otra es una validación que no necesita pedirle nada a
nadie, y es la que encuentra los tres errores de kilómetros que ninguna de las
comprobaciones anteriores podía ver.

Cada paso trae el Pandas, su salida real, y la fórmula de Excel dibujada en su hoja. Diez
de los doce traen además **la trampa**: lo que sale mal si el paso se hace de la forma
obvia, que es la parte que no está en ningún tutorial y la que se pregunta en una prueba.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.bip_lugares_carga import solution as p01
from lab.excel import detect_header_row, read_all_sheets, sheet_names
from lab.geo import haversine_km, utm_to_latlon
from lab.textual import normalize_key

import pandas as pd

ORIGINAL = p01.ORIGINAL
ruta_metro = ORIGINAL / 'bip-metro.xlsx'
ESQUEMA = list(p01.ESQUEMA)

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Mirar el archivo antes de leerlo


Cuántas hojas tiene cada archivo y cómo se llaman. Uno tiene dos, y la segunda son locales **fuera de servicio**.

> ⚠️ `pd.read_excel(ruta)` sin `sheet_name` lee **solo la primera hoja** y no avisa. Y las hojas no se llaman igual en los cinco archivos —`Abiertos`, `Abierto`, `PCMA`—, de modo que tampoco sirve pedirla por nombre.


In [ ]:
{archivo.stem: sheet_names(archivo)
 for archivo in sorted(ORIGINAL.glob('bip-*.xlsx'))}


### Paso 2 · Encontrar la fila del encabezado


No está en la fila 1: está en la 7, 9, 13, 10 y 9, una por archivo. Con `header=0` las columnas salen como `Unnamed: 0`.

> ⚠️ No sirve «la primera fila sin celdas vacías»: una fila de datos también lo cumple. Hay que exigir varias celdas de **texto** seguidas de filas con datos.


In [ ]:
crudo = pd.read_excel(ruta_metro, sheet_name='Abiertos', header=None)
detect_header_row(crudo)


### Paso 3 · Leer las seis hojas con su encabezado


El informe de lectura no es decorativo: es lo que permite afirmar después que el 1 928 no se inventó.


In [ ]:
hojas = p01.cargar()
p01.informe_de_lectura(hojas)


### Paso 4 · Los esquemas no coinciden


Nueve columnas en unos archivos y diez en otros, y el mismo campo con dos nombres: `NOMBRE FANTASIA` y `NOMBRE DE FANTASIA`.

**En Excel:** `=COINCIDIR(A1;$B$1:$B$10;0)`

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Una columna trae un **espacio al final del nombre**. Es invisible, y un renombrado por igualdad exacta falla sin error: la columna queda fuera del esquema.


In [ ]:
p01.informe_de_esquemas(hojas)


### Paso 5 · Apilar los cinco al esquema común


La correspondencia está **declarada** en el diccionario `ALIAS`, no deducida por parecido.


In [ ]:
consolidado = p01.consolidar(hojas)
consolidado[ESQUEMA]


### Paso 6 · El tipo de local no está en los datos


Está en el **nombre del archivo**. Si no se agrega al apilar, se pierde.

> ⚠️ Es el error más fácil de no notar: la tabla queda completa, sumable y sin huecos, y ya no se puede responder «¿cuántos son de Metro?».


In [ ]:
consolidado.groupby('tipo_local').size().sort_values(ascending=False)


### Paso 7 · ¿La columna «código» es una clave?


Medirlo cuesta una línea, y hay que hacerlo antes de cruzar.

**En Excel:** `=SUMAPRODUCTO((CONTAR.SI(A2:A1929;A2:A1929)>1)*1)`

Debe dar: **136 filas con código repetido, de 1 928**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ En el archivo de Metro el código es la **línea** (`L1`, `L5`), no el local: 143 estaciones con 7 códigos. Y tampoco sirve `archivo + código`.


In [ ]:
{'filas': len(consolidado),
 'distintos': consolidado['codigo'].nunique(),
 'repetidos': int(consolidado['codigo'].duplicated().sum())}


### Paso 8 · Normalizar antes de agrupar


`FULLCARGA` y `Fullcarga` son dos grupos para `groupby` y uno para una persona.

**En Excel:** `=MAYUSC(ESPACIOS(B2))`

Debe dar: **4 valores crudos, 3 normalizados**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Guardar **las dos** formas: la clave para agrupar y la etiqueta para mostrar. Con solo la clave, el informe dice `NUNOA` en vez de `Ñuñoa`.


In [ ]:
{'crudas': consolidado['entidad'].nunique(),
 'normalizadas': consolidado['entidad'].map(normalize_key).nunique()}


### Paso 9 · Contar por comuna


La pregunta que se pedía, con el desglose por tipo de local.

**En Excel:** `=CONTAR.SI.CONJUNTO($E$2:$E$1929;$N2;$M$2:$M$1929;O$1)`

Debe dar: **Maipú 157, Santiago 148, Las Condes 104**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ `groupby` **descarta los nulos de la clave** sin avisar, y el total deja de sumar las filas. Con `dropna=False` aparecen como grupo propio.


In [ ]:
p01.por_comuna(consolidado)


### Paso 10 · Cuánto del campo «horario» existe


Antes de usar un campo, saber cuánto está lleno. Y declarar **sobre qué**: la cobertura cambia según se cuenten las 1 928 filas o las 1 927 abiertas.

**En Excel:** `=CONTAR.SI(F2:F1929;"?*")/FILAS(F2:F1929)`

Debe dar: **18,10 % sobre las 1 928 filas · 18,06 % sobre las 1 927 abiertas**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Dos trampas en el mismo paso. Una: `CONTARA` y `CONTAR.SI(...;"<>")` cuentan como no vacía una celda con **fórmula que devuelve cadena vacía**, de modo que una columna traída por una consulta da 100 % estando vacía; `"?*"` exige al menos un carácter. Otra: el **denominador es una decisión**. 18,10 % y 18,06 % son la misma medición sobre dos poblaciones distintas, y hay que decir cuál se publica.


In [ ]:
def cobertura(tabla):
    lleno = tabla['horario'].notna() & (
        tabla['horario'].astype('string').str.strip() != '')
    return round(float(lleno.mean() * 100), 2)

{'todas las filas': cobertura(consolidado),
 'solo abiertas': cobertura(consolidado[consolidado['abierto']])}


### Paso 11 · Las dos coordenadas de cada fila


El archivo trae la misma ubicación en UTM y en grados. Convertir una y comparar con la otra es una validación que no necesita fuente externa.

**En Excel:** `=FRECUENCIA(N2:N1929;P1:P6)`

Debe dar: **109 filas bajo 10 m y 1 768 entre 58 y 62 m**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ La media y los percentiles 50, 90 y 99 **esconden el hallazgo**: los tres caen en 61 m y parece un desvío único con tres atípicos. El histograma muestra **dos grupos**, que es la firma de dos datums distintos y no de un redondeo. Un resumen de cinco números no los distingue.


In [ ]:
lat_utm, lon_utm = utm_to_latlon(
    consolidado['este'], consolidado['norte'], zona=19)
desvio_m = haversine_km(
    lat_utm, lon_utm,
    consolidado['latitud'], consolidado['longitud']) * 1000
pd.cut(desvio_m, [0, 10, 50, 58, 62, 100, 1e9]).value_counts().sort_index()


### Paso 12 · Las filas donde las dos coordenadas se contradicen


Con el desvío calculado, un umbral declarado separa el desplazamiento sistemático de los errores reales.

**En Excel:** `=CONTAR.SI(N2:N1929;">200")`

Debe dar: **3 filas, con desvíos de 893 m a 10,3 km**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ El umbral es una **decisión** y se escribe con su motivo. Elegirlo después de ver los datos, para que el número de hallazgos quede cómodo, es el resultado disfrazado de criterio.


In [ ]:
p01.coordenadas_sospechosas(consolidado)[
    ['fuente', 'codigo', 'comuna', 'desvio_m']]


## 4 · El resultado

1 928 filas, 1 927 abiertas, 46 comunas, 3 entidades. El `codigo` no identifica una fila, ni junto al archivo de origen. La validación cruzada de las dos coordenadas encontró 3 errores de hasta 10,3 km y un desplazamiento sistemático de 61 m con distribución bimodal.

**No** se puede concluir cuánta gente usa cada local —no hay transacciones— ni si está abierto cuando alguien llega —el horario está en el 18,06 % de las filas—.

## 5 · Qué queda aprendido

1. El encabezado de una planilla real **no está en la primera fila**, y la fila se detecta en lugar de fijarse: así el código sobrevive a la versión siguiente.
2. Una correspondencia de columnas se **declara**. Deducirla por parecido acierta hoy y falla cuando cambie un nombre, sin avisar.
3. Lo que no está en ninguna columna —aquí, el tipo de local— se pierde al apilar si no se agrega a propósito. Y la tabla resultante queda completa y sumable.
4. Una columna llamada «código» **no es una clave hasta que se mide**, y en este archivo no lo es ni junto al nombre del archivo de origen.
5. Normalizar es decidir qué cosas son la misma, y hay que guardar **las dos** formas: la clave para agrupar y la etiqueta para mostrar.
6. El **denominador es una decisión**: la cobertura del horario es 18,10 % o 18,06 % según se cuenten todas las filas o solo las abiertas, y hay que decir cuál se publica.
7. Un campo redundante —la misma ubicación dos veces— es una validación gratuita que no necesita ninguna fuente externa.
8. Un resumen de cinco números **esconde la forma**: la media y los percentiles no distinguen dos sistemas de coordenadas de un error de redondeo. El histograma, sí.
9. Un umbral se escribe con su motivo. Elegirlo después de ver los datos, para que el número de hallazgos quede cómodo, es el resultado disfrazado de criterio.
10. Lo que corresponde con un error de datos ajenos es **reportarlo al publicador**, no corregirlo en silencio: no se sabe cuál de las dos versiones es la buena.
